# CiteAgent VN trên Kaggle

Chạy lại toàn bộ pipeline (snapshot, index, bộ xác minh vùng xám) trên GPU Kaggle với một model Ollama lớn hơn
laptop. **Không dùng API key nào.**

Trước khi chạy:
1. *Settings → Accelerator*: **GPU T4 x2** (hoặc P100). *Settings → Internet*: **On**.
2. (Khuyến nghị) *Add Input*: dataset riêng tư chứa `citeagent_sources.zip` tạo bằng
   `python scripts/pack_sources.py --with-embeddings` trên máy của bạn.
3. Sửa `VERIFIER_MODEL` ở ô dưới nếu muốn (qwen3:8b ~5 GB, qwen3:14b ~9 GB VRAM).

Kết quả: `/kaggle/working/citeagent_results.zip` (thư mục `reports/`) trong mục Output.

In [ ]:
!nvidia-smi
!python --version

In [ ]:
%cd /kaggle/working
!git clone --depth 1 https://github.com/ThanhDatVN/chatbot_legal.git
%cd /kaggle/working/chatbot_legal
!git log --oneline -1

In [ ]:
import glob, os
os.environ['VERIFIER_MODEL'] = 'qwen3:8b'
found = glob.glob('/kaggle/input/**/citeagent_sources.zip', recursive=True)
os.environ['SOURCES_ZIP'] = found[0] if found else ''
print('VERIFIER_MODEL =', os.environ['VERIFIER_MODEL'], '| SOURCES_ZIP =', os.environ['SOURCES_ZIP'] or '(none: download from CDN)')

In [ ]:
!bash kaggle/run_on_kaggle.sh 2>&1 | tee /kaggle/working/run.log

In [ ]:
import json, glob
for path in sorted(glob.glob('reports/verifier_tuning_*.json')):
    print(path, json.load(open(path, encoding='utf-8'))['best'])